In [1]:
ABLATION_CHANNELS = {
    # Axis ablation
    "x": [0, 3, 6, 9, 12, 15],
    "y": [1, 4, 7, 10, 13, 16],
    "z": [2, 5, 8, 11, 14, 17],

    # Signal-type ablation
    "accelerometer": [0, 1, 2],
    "gravity": [3, 4, 5],
    "gyros": [6, 7, 8],
    "lin_accel": [9, 10, 11],
    "game_rot_vec": [12, 13, 14],
    "magn_field": [15, 16, 17]
}

In [7]:
import sys
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

from torch import Tensor, concat
from utils import input_generation


experiments = [None] + list(ABLATION_CHANNELS.keys())

results = []


for ablation in experiments:

    X, y, channel_names = input_generation(Path("KSAS-Dataset/movements"), ablation, ABLATION_CHANNELS)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


    #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    #-----------------
    # X-ROCKET ENCODER
    #-----------------
    repo_path = str(Path.cwd() / "xrocket")
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    from xrocket.encoder import XRocket

    encoder = XRocket(in_channels=len(channel_names), max_kernel_span=33, combination_order=1, feature_cap=10_000, kernel_length=9, max_dilations=32)
    encoder.fit(Tensor(X_train[0]).unsqueeze(0))


    #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    embed_train, embed_test = [], []

    for x in X_train:
        embed_train.append(encoder(Tensor(x).unsqueeze(0)))
    for x in X_test:
        embed_test.append(encoder(Tensor(x).unsqueeze(0)))


    feature_names = pd.Index(encoder.feature_names).rename(["pattern", "dilation", "channel", "threshold"])
    embed_train = pd.DataFrame(data=concat(embed_train), columns=feature_names)
    embed_test = pd.DataFrame(data=concat(embed_test), columns=feature_names)


    #-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    #-----------
    # PREDICTION
    #-----------
    model = RandomForestClassifier(random_state=42)
    model.fit(embed_train, y_train)


    # accuracy score
    pred_test = model.predict(embed_test)
    acc_test = accuracy_score(y_test, pred_test)


    for movement in sorted(set(y_test)):

        mask = y_test == movement

        movement_accuracy = accuracy_score(y_test[mask], pred_test[mask])

        results.append({
            "ablation": "baseline" if ablation is None else ablation,
            "removed_channels": 0 if ablation is None else len(ABLATION_CHANNELS[ablation]),
            "remaining_channels": len(channel_names),
            "movement": movement,
            "accuracy": movement_accuracy
        })


## RESULTS VISUALIZATION

In [6]:
import matplotlib.pyplot as plt


movement_names = [
    "No movement",
    "Upward",
    "Hammering inward",
    "Extended outward",
    "Outward downward",
    "Rear elbow"
]

cm = confusion_matrix(y_test, pred_test)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=movement_names)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, cmap="viridis", values_format="d", colorbar=True)

plt.xticks(rotation=45, ha="right")

ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix")

plt.tight_layout()
plt.show()

NameError: name 'confusion_matrix' is not defined

# Task 1.1

In [3]:
results_df = pd.DataFrame(results)

accuracy_table = results_df.pivot(index="ablation", columns="movement", values="accuracy")
accuracy_table

movement,0,1,2,3,4,5
ablation,,,,,,
accelerometer,1.000000,0.916667,0.750000,0.583333,0.833333,0.916667
baseline,1.000000,0.916667,0.750000,0.750000,0.833333,1.000000
game_rot_vec,0.916667,0.916667,0.833333,0.583333,0.833333,1.000000
gravity,1.000000,1.000000,0.666667,0.500000,0.833333,1.000000
gyros,1.000000,1.000000,0.750000,0.583333,0.833333,1.000000
lin_accel,0.916667,1.000000,0.750000,0.500000,0.833333,1.000000
magn_field,1.000000,1.000000,0.750000,0.500000,0.833333,1.000000
x,1.000000,1.000000,0.750000,0.500000,0.833333,1.000000
y,1.000000,1.000000,0.750000,0.583333,0.666667,0.916667


In [4]:
baseline = accuracy_table.loc["baseline"]

delta_table = accuracy_table.subtract(baseline, axis=1)

delta_table = delta_table.drop("baseline")

delta_table

movement,0,1,2,3,4,5
ablation,,,,,,
accelerometer,0.000000,0.000000,0.000000,-0.166667,0.000000,-0.083333
game_rot_vec,-0.083333,0.000000,0.083333,-0.166667,0.000000,0.000000
gravity,0.000000,0.083333,-0.083333,-0.250000,0.000000,0.000000
gyros,0.000000,0.083333,0.000000,-0.166667,0.000000,0.000000
lin_accel,-0.083333,0.083333,0.000000,-0.250000,0.000000,0.000000
magn_field,0.000000,0.083333,0.000000,-0.250000,0.000000,0.000000
x,0.000000,0.083333,0.000000,-0.250000,0.000000,0.000000
y,0.000000,0.083333,0.000000,-0.166667,-0.166667,-0.083333
z,0.000000,-0.083333,-0.083333,-0.333333,0.000000,0.000000


In [ ]:
dilation_importances = feature_importances.groupby("dilation").sum().sort_values()
ax = dilation_importances.plot(kind="barh")

In [ ]:
pattern_importances = feature_importances.groupby("pattern").sum().sort_values()
ax = pattern_importances.plot(kind="barh", figsize=(6,20))
